In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from typing import TypedDict, Annotated
from dotenv import load_dotenv

In [2]:
load_dotenv()

model = ChatOpenAI()

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state:ChatState):
    response = model.invoke(state['messages'])
    return {'messages':[response]}

In [25]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)

In [26]:
# chatbot.invoke({'messages':[HumanMessage(content="What is the capital of illinois")]})

In [31]:
thread_id='1'
while True:
    user_message = input("Type here: ")
    print("User: ",user_message)
    if user_message.strip().lower() in ["quit","bye","exit"]:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = chatbot.invoke({'messages':HumanMessage(content=user_message)},config=config)
    print('AI: ',response['messages'][-1].content)

User:  What is your name
AI:  My name is Assistant. How can I assist you today?
User:  My name is Ronaldo
AI:  Nice to meet you, Ronaldo! How can I assist you today?
User:  Who's the most famous person you know with my name
AI:  The most famous person I can think of with the name Ronaldo is Cristiano Ronaldo, the famous Portuguese soccer player who currently plays for Manchester United.
User:  Does he really play for United? Think hard
AI:  I apologize for the mistake in my previous response. As of my last update, Cristiano Ronaldo was playing for Juventus. Thank you for pointing that out!
User:  Still Wrong
AI:  I apologize for the oversight. As of September 2021, Cristiano Ronaldo rejoined Manchester United. Thank you for your patience.
User:  Today is 2026 dawg
AI:  I apologize for the outdated information. As of 2026, Cristiano Ronaldo's team affiliation may have changed. I recommend checking the most recent sources for accurate updates on his current team. Thank you for bringing t